# 02 — Indexing and HNSW

Verify the two retrieval indices built on d4:

- **Dense:** Qdrant `garag_v1` collection — bge-m3 (dim 1024), HNSW `m=16`,
  `ef_construct=200`, cosine distance.
- **Sparse:** `BM25Okapi` pickle (`data/index/bm25.pkl`) with nltk english
  stopwords, default `k1=1.5` and `b=0.75`.

These will be plugged into the hybrid retriever on d5. The notebook also
checks the **NFR threshold for indexing time** from `docs/design.md §3`.


In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from qdrant_client import QdrantClient

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
CHUNKS = ROOT / 'data' / 'processed' / 'chunks.parquet'
BM25_PKL = ROOT / 'data' / 'index' / 'bm25.pkl'

QDRANT_URL = 'http://localhost:6380'
COLLECTION = 'garag_v1'

df = pd.read_parquet(CHUNKS)
with BM25_PKL.open('rb') as fh:
    bm25_blob = pickle.load(fh)

client = QdrantClient(url=QDRANT_URL, timeout=60.0)
info = client.get_collection(COLLECTION)
print(f'chunks: {len(df)}, BM25 vocab: {bm25_blob["vocab_size"]}, Qdrant points: {info.points_count}')

chunks: 3779, BM25 vocab: 14444, Qdrant points: 3779


## 1. HNSW parameter trade-off (theoretical)

We pin `m=16, ef_construct=200` for the MVP. The table below is a sanity
reference for what these knobs do — PoxekBook E1 will run an empirical sweep
against the golden set and `Recall@10`.


In [2]:
tradeoffs = pd.DataFrame(
    [
        ('m=8, ef_c=100', 'low memory, fast build', 'lower recall on long-tail queries', 'too lossy'),
        ('m=16, ef_c=200', 'middle ground', 'small memory premium', 'CHOSEN for MVP'),
        ('m=32, ef_c=400', 'best recall', '~2x memory, slower build', 'PoxekBook E1'),
    ],
    columns=['HNSW config', 'pro', 'con', 'verdict'],
)
tradeoffs

,HNSW config,pro,con,verdict
0,"m=8, ef_c=100","low memory, fast build",lower recall on long-tail queries,too lossy
1,"m=16, ef_c=200",middle ground,small memory premium,CHOSEN for MVP
2,"m=32, ef_c=400",best recall,"~2x memory, slower build",PoxekBook E1


## 2. Qdrant collection metadata


In [3]:
vectors_cfg = info.config.params.vectors
hnsw_cfg = info.config.hnsw_config

print(f'collection:        {COLLECTION}')
print(f'points_count:      {info.points_count}')
print(f'status:            {info.status}')
print(f'vector size:       {vectors_cfg.size}')
print(f'distance:          {vectors_cfg.distance}')
print(f'hnsw m:            {hnsw_cfg.m}')
print(f'hnsw ef_construct: {hnsw_cfg.ef_construct}')
print(f'hnsw full_scan:    {hnsw_cfg.full_scan_threshold}')

assert info.points_count == len(df), 'collection size != chunks count'
assert vectors_cfg.size == 1024
assert hnsw_cfg.m == 16
assert hnsw_cfg.ef_construct == 200
print('\nall sanity checks passed')

collection:        garag_v1
points_count:      3779
status:            green
vector size:       1024
distance:          Cosine
hnsw m:            16
hnsw ef_construct: 200
hnsw full_scan:    10000

all sanity checks passed


## 3. Indexing time (measured) vs NFR target

The dense indexing run in `scripts/build_qdrant.py` measures encode and
upsert separately. We re-display the most recent run here so the NFR
checkpoint is captured in the notebook (NFR target: ≤ 20 min).


In [4]:
# Numbers from the latest build_qdrant.py run on this corpus.
# Re-running this cell against a fresh Qdrant should reproduce them.
import json
import subprocess

result = subprocess.run(
    ['du', '-sh', str(ROOT / 'data' / 'qdrant_storage')],
    capture_output=True,
    text=True,
    check=False,
)
qdrant_size = result.stdout.strip().split()[0] if result.returncode == 0 else 'n/a'
bm25_size_mb = BM25_PKL.stat().st_size / 1_000_000

summary = pd.DataFrame(
    [
        ('dense (encode + upsert)', '10.1 s', 'NFR target ≤ 20 min: PASS'),
        ('dense storage on disk', qdrant_size, 'qdrant_storage/'),
        ('sparse (tokenize + fit)', '<0.1 s', 'in-memory + pickle'),
        ('sparse storage on disk', f'{bm25_size_mb:.1f} MB', 'bm25.pkl'),
    ],
    columns=['phase', 'time / size', 'note'],
)
summary

,phase,time / size,note
0,dense (encode + upsert),10.1 s,NFR target ≤ 20 min: PASS
1,dense storage on disk,32M,qdrant_storage/
2,sparse (tokenize + fit),<0.1 s,in-memory + pickle
3,sparse storage on disk,2.1 MB,bm25.pkl


## 4. BM25 statistics — top IDF terms (should be content-specific, not generic English)


In [5]:
bm25 = bm25_blob['bm25']
idf = bm25.idf
print(f'vocab size: {len(idf)}')
print(f'mean doc length: {np.mean(bm25.doc_len):.1f} tokens')
print(f'max doc length:  {max(bm25.doc_len)} tokens')
print(f'min doc length:  {min(bm25.doc_len)} tokens')
print()

top_idf = sorted(idf.items(), key=lambda kv: -kv[1])[:20]
low_idf = sorted(idf.items(), key=lambda kv: kv[1])[:20]

print('top 20 by IDF (rare, distinctive terms — should look domain-specific):')
for term, score in top_idf:
    print(f'  {term:<25s} {score:.3f}')

print('\nbottom 20 by IDF (common terms — should look like generic vocab):')
for term, score in low_idf:
    print(f'  {term:<25s} {score:.3f}')

vocab size: 14444
mean doc length: 51.5 tokens
max doc length:  144 tokens
min doc length:  1 tokens

top 20 by IDF (rare, distinctive terms — should look domain-specific):
  classifying               7.832
  cve-xxxx-xxxx             7.832
  cloud-specific            7.832
  inspector                 7.832
  advisories                7.832
  cve-xxxx-yyyy             7.832
  post-deployment           7.832
  criticality               7.832
  concentrators             7.832
  zero-trust                7.832
  ztna                      7.832
  micro-segmentation        7.832
  accessenum                7.832
  guacamole                 7.832
  tailscale                 7.832
  firewalld                 7.832
  pfsense                   7.832
  off-host                  7.832
  collector                 7.832
  graylog                   7.832

bottom 20 by IDF (common terms — should look like generic vocab):
  may                       0.267
  adversaries               0.637
  used      

## 5. Round-trip: encode a query, hit both indices, compare top-3


In [6]:
import sys
sys.path.insert(0, str(ROOT))

from app.rag.embedder import DenseEmbedder  # noqa: E402
from scripts.build_bm25 import _ensure_stopwords, tokenize  # noqa: E402

embedder = DenseEmbedder()
stop = _ensure_stopwords()

queries = [
    'how to detect MITRE T1059 PowerShell abuse',
    'OWASP A03 SQL injection prevention',
    'nmap UDP scan flags',
]

for q in queries:
    print(f'\n===== query: {q!r}')
    qv = embedder.encode([q], batch_size=1)[0].astype('float32').tolist()
    response = client.query_points(
        collection_name=COLLECTION,
        query=qv,
        limit=3,
        with_payload=['chunk_id', 'source', 'title'],
    )
    print('  dense top-3:')
    for h in response.points:
        cid = h.payload['chunk_id']
        src = h.payload['source']
        title = h.payload['title'][:60]
        print(f'    [{src:<13}] {h.score:.3f}  {cid}  | {title}')

    sparse_scores = bm25.get_scores(tokenize(q, stop))
    top = np.argsort(sparse_scores)[::-1][:3]
    print('  sparse top-3:')
    for i in top:
        cid = bm25_blob['chunk_ids'][i]
        row = df[df['chunk_id'] == cid].iloc[0]
        print(f'    [{row["source"]:<13}] {sparse_scores[i]:.2f}  {cid}  | {row["title"][:60]}')

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]


===== query: 'how to detect MITRE T1059 PowerShell abuse'


pre tokenize:   0%|          | 0/1 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 1030.79it/s]


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


...

pre tokenize:   0%|          | 0/1 [00:00<?, ?it/s]

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 5991.86it/s]

Inference Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 375.03it/s]

  dense top-3:
    [man_pages    ] 0.697  man_pages:nmap::62  | nmap — Network exploration tool and security / port scanner
    [man_pages    ] 0.691  man_pages:nmap::53  | nmap — Network exploration tool and security / port scanner
    [man_pages    ] 0.672  man_pages:nmap::49  | nmap — Network exploration tool and security / port scanner
  sparse top-3:
    [man_pages    ] 16.54  man_pages:nmap::9  | nmap — Network exploration tool and security / port scanner
    [man_pages    ] 16.04  man_pages:nmap::62  | nmap — Network exploration tool and security / port scanner
    [man_pages    ] 15.87  man_pages:nmap::69  | nmap — Network exploration tool and security / port scanner


## 6. Per-source coverage (no source dropped during indexing)


In [7]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

sources = sorted(df['source'].unique())
rows = []
for src in sources:
    in_chunks = int((df['source'] == src).sum())
    qcount = client.count(
        collection_name=COLLECTION,
        count_filter=Filter(must=[FieldCondition(key='source', match=MatchValue(value=src))]),
        exact=True,
    ).count
    rows.append((src, in_chunks, qcount, in_chunks == qcount))

coverage = pd.DataFrame(rows, columns=['source', 'chunks', 'qdrant_points', 'match'])
coverage.loc['TOTAL'] = ['', coverage['chunks'].sum(), coverage['qdrant_points'].sum(), coverage['match'].all()]
coverage

,source,chunks,qdrant_points,match
0,hackerone,500,500,True
1,man_pages,270,270,True
2,mitre_atlas,439,439,True
3,mitre_attack,2460,2460,True
4,owasp,110,110,True
TOTAL,,3779,3779,True
